# Digital Cash Management Forecast: set up the project

This notebook sets up the Digital Cash Management Forecast in this Dataiku project (it also gives the project
that name), the way it runs in production. Everything the project writes is a CSV file. It reads 2 warehouse tables
as they are, read-only, and creates nothing in the warehouse: the clients (`EDW.DIM_CUSTOMER`) and the daily
postings (`EDW.FCT_STATEMENT_ENTRY`), which the staging recipe sums per month in Dataiku. In the sandbox, they are
the datasets the notebook `warehouse_simulator.ipynb` wrote in this project; in production, the warehouse's tables
through the read-only database connection `WAREHOUSE_SQL_CONNECTION` (or CSV files in the landing folder). Names and
codes: `python/warehouse.py` in the
project library). It forecasts one value per client and month: the net cash flow (money in minus money out), 1,
2 and 3 months ahead, as P10, EV and P90; there are no categories. It builds 5 flow zones, each with an owner:

- **Raw data** and **Staging** (shared): the 2 warehouse tables, read-only, then the business clients and the
  postings summed per client and month, with the standard names and values.
- **Training** (data scientist): the training rows, then each training run is one model with its own **model ID**,
  trained with the **forecaster** the project variable `forecaster` names, tracked in Dataiku's **Experiment
  Tracking** and deployed as a new **version** of the saved models `forecast_ev`, `forecast_p10`, `forecast_p90`.
  A champion / challenger test activates the new version if it beats the production model.
- **Inference** (inference owner): the scoring rows (latest month), visual Score recipes with the active version, the 3-month
  forecast (every row says which model and which load produced it), the forecast archive, and the inference log
  (one row per run: when, which model, which load, how many clients).
- **Lab** (data scientist): `try_forecaster` tries the forecaster the project variable `lab_forecaster` names
  against the production model and adds the result to `lab_results`, without touching production. The notebook
  `forecast_lab.ipynb` does the same interactively.

The zones only meet through three interfaces: **1. the model rows** (`model_rows` in `cashflow.py`: staging →
training and inference), **2. the forecaster** (`models.py`: lab → training), **3. the active version of the saved
models** (training → inference). Two independent scenarios run it: **Nightly batch** (inference) and **Retrain,
test and promote** (training). Each starts by setting the cut-off month (`set_cutoff`: the month before today, or
the project variable `cutoff_override`); the staging recipe stops if the postings do not reach it.

It then trains, tests and promotes a first model, and runs the first nightly batch.

**How to use it:** in the project, import this notebook and `warehouse_simulator.ipynb` (Notebooks > New notebook
> Import). In the sandbox, run the simulator first. Then check the settings below and run all cells (Cell > Run All):
a few minutes with the small sample data. It can be run again in the same project: what exists is kept (data,
saved model versions, zones, scenario history), and the recipes' code is updated.

In [ ]:
PROJECT_NAME = "Digital Cash Management Forecast"
WAREHOUSE_SQL_CONNECTION = ""         # production: the read-only database connection to the warehouse (Oracle), to
WAREHOUSE_SQL_SCHEMA = "EDW"          # read DIM_CUSTOMER and FCT_STATEMENT_ENTRY of this schema as they are. Empty:
LANDING_CONNECTION = "EDW_LANDING"    # read them as CSV files from the landing folder (read-only connection). Neither
                                      # is used for tables the warehouse simulator has written here (sandbox: CSV).
CONNECTION = "filesystem_managed"     # where the project writes its own datasets (CSV files)
FOLDER_CONNECTION = "filesystem_folders"   # where the tracked models and the forecast archive are kept
CODE_ENV = "cashflow_forecast"        # Python code env for the recipes: an existing one (an admin creates it once)

## 1. Code environment

The recipes need CatBoost, and MLflow for Dataiku's Experiment Tracking, in an existing code env (`CODE_ENV`
above). This notebook does not create it: an admin creates it once. This checks it is there with those packages,
and otherwise stops with the request to send the admin.

In [ ]:
import dataiku

client = dataiku.api_client()
project = client.get_project(dataiku.default_project_key())
# catboost: the models; mlflow: Experiment Tracking and the saved models. Dataiku tested MLflow models with
# mlflow 2.22.0 and 3.10.1 (DSS 14 and 15 docs, "MLflow models: limitations"); mlflow 3 needs Python 3.10 or later.
# Either works here; the request below asks for 2.22.0, which also runs on Python 3.9.
PACKAGES = ("catboost==1.2.7", "scikit-learn", "mlflow==2.22.0")


def check_code_env(name, packages=PACKAGES):
    """Check the code env exists and lists the packages. It is not created here: an admin creates it once."""
    request = (f"Ask a Dataiku admin to create the Python code env {name} (Python 3.9 or later) with these "
               f"packages: {', '.join(packages)}. Or set CODE_ENV to an existing env that has them.")
    if name not in [e["envName"] for e in client.list_code_envs()]:
        raise RuntimeError(f"Code env {name} not found. {request}")
    try:
        listed = " ".join(client.get_code_env("PYTHON", name).get_settings().get_required_packages(as_list=True))
    except Exception:          # no right to read the env settings: it is checked when the first recipe runs
        print(f"Code env {name} found (could not read its packages)")
        return
    missing = [p for p in packages if p.split("=")[0].split(">")[0] not in listed]
    if missing:
        raise RuntimeError(f"Code env {name} lacks {', '.join(missing)}. {request}")
    print(f"Code env {name} has {', '.join(packages)}")


check_code_env(CODE_ENV)

## 2. Code

The project library gets these files, by owner:

- shared: `warehouse.py` (the warehouse tables and columns read, codes, monthly sums, cut-off month) and
  `cashflow.py` (dates, the model rows, the saved model names)
- training and lab: `models.py` (the forecaster interface, the reference CatBoost forecaster, the test and its
  scores) and `tracking.py` (model IDs, Experiment Tracking, saved model versions, champion / challenger, promotion)
- inference: `forecast.py` (the production model's ID, the final forecast table)
- `forecasters.py`: the data scientist's own forecasters. Written only when it is created: running this notebook
  again never overwrites it.

Each recipe below becomes a Python recipe in the flow; the visual recipes are created in the next step.

In [ ]:
WAREHOUSE_CODE = r'''
"""Warehouse tables (project library: python/warehouse.py).

The forecast reads 2 tables of the bank's warehouse as they are, read-only, and creates nothing there:
    DIM_CUSTOMER          EDW.DIM_CUSTOMER: the clients. The project keeps the business clients (companies and sole
                          proprietors) and only the columns below: personal fields are never read.
    FCT_STATEMENT_ENTRY   EDW.FCT_STATEMENT_ENTRY: the daily postings. The staging recipe reads them in chunks and
                          sums them per client, month and direction (money in, money out) in Dataiku: no query
                          runs in the warehouse but reading the rows.
Through the database connection to the warehouse (Oracle, read-only), or as CSV files in a landing folder; the setup
notebook's settings choose. The sandbox's simulated tables are CSV. If a name or a code differs at the bank, only
this file changes.
"""
import numpy as np
import pandas as pd

# the tables the project reads. standard name: (table, where it is)
TABLES = {
    "clients": ("DIM_CUSTOMER", "EDW.DIM_CUSTOMER"),
    "transactions": ("FCT_STATEMENT_ENTRY", "EDW.FCT_STATEMENT_ENTRY"),
}
STAGED = ["clients", "monthly_flows"]     # what the staging recipe writes
HISTORY_MONTHS = 60            # months of postings summed, up to the cut-off month

# the columns read. standard name: warehouse (EDW) name
COLUMNS = {
    "clients": {"client_no": "CSTMR_CIF", "customer_type": "CSTMR_TYP", "line_of_business": "LOB_DSC",
                "legal_entity": "LGL_ENTY_DSC", "sub_legal_entity": "PRTY_SUB_LGL_DSC", "id_type": "PRTY_ID_TYP_DSC",
                "segment": "RM_SGMNT_NM", "sector": "SCTR_DSC", "business_sector": "BUS_SCTR_EN_NM",
                "industry": "INDST_EN_NM", "sub_industry": "SUB_INDST_EN_NM", "city": "CITY", "status": "STS_FLG",
                "last_annual_sales": "ANNL_SLS_AMT", "number_of_employees": "NMBR_OF_EMP"},
    # one row per posting. AMT_LCY: SAR, negative for money out (T24). RVRSL_MRKR: set on a reversing posting, which
    # carries the reversed posting's TRNS_RFR
    "transactions": {"entry_id": "STMT_ENTRY_ID", "client_no": "CSTMR_CIF", "account_no": "ACNT_NMBR",
                     "value_date": "VAL_DT", "direction": "CRDT_DBT_FLG", "amount": "AMT_LCY", "txn_ref": "TRNS_RFR",
                     "reversal": "RVRSL_MRKR"},
}
OPTIONAL = {"ANNL_SLS_AMT", "NMBR_OF_EMP"}    # not in EDW.DIM_CUSTOMER today: read if they appear, else empty
# codes in DIM_CUSTOMER (from the bank's data profile; to confirm on business clients)
ACTIVE_STATUS = ("ACTIVE", "A")               # STS_FLG of an active client (an inactive one shows INACTIVE)
RETAIL = ("RETAIL",)                          # words of LOB_DSC for retail (individual) clients: left out...
BUSINESS_ID = ("COMMERCIAL",)                 # ...unless their ID type (PRTY_ID_TYP_DSC) is a commercial registration
SOLE_PROPRIETOR = ("SOLE", "ESTABLISHMENT")   # words of LGL_ENTY_DSC or PRTY_SUB_LGL_DSC for a business owned by 1 person


def read_columns(table):
    """The warehouse columns the project reads from a table (the optional ones are read only if they exist)."""
    return list(COLUMNS[table].values())


def to_standard(df, table):
    """A warehouse table with the standard column names and the project's values: client numbers as text, direction
    IN / OUT; for clients, status ACTIVE / CLOSED and sole_proprietor true / false. An optional column that is not
    there stays empty; a missing required column stops here with its name."""
    missing = [c for c in COLUMNS[table].values() if c not in df and c not in OPTIONAL]
    if missing:
        raise ValueError(f"{TABLES[table][0]} has no column {', '.join(missing)}: check the warehouse table")
    out = pd.DataFrame({standard: df[column] if column in df else np.nan for standard, column in COLUMNS[table].items()})
    out["client_no"] = out["client_no"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    if "direction" in out:
        out["direction"] = out["direction"].astype(str).str.strip().str.upper().str[0].map({"C": "IN", "D": "OUT"})
    if table == "clients":
        active = out["status"].astype(str).str.strip().str.upper().isin(ACTIVE_STATUS)
        out["status"] = np.where(active, "ACTIVE", "CLOSED")
        legal = (out["legal_entity"].fillna("").astype(str) + " " + out["sub_legal_entity"].fillna("").astype(str))
        out["sole_proprietor"] = legal.str.upper().str.contains("|".join(SOLE_PROPRIETOR))
    return out


def business_clients(clients):
    """The business clients (standard names): every client whose line of business does not contain RETAIL, plus the retail
    ones registered with a commercial registration (sole proprietors). One row per client number."""
    lob = clients["line_of_business"].fillna("").astype(str).str.upper()
    id_type = clients["id_type"].fillna("").astype(str).str.upper()
    business = ~lob.str.contains("|".join(RETAIL)) | id_type.str.contains("|".join(BUSINESS_ID))
    return clients[business].drop_duplicates("client_no", keep="last").reset_index(drop=True)


def monthly_flows(entries, client_numbers, cutoff):
    """The monthly totals of postings (standard names): business clients only, the HISTORY_MONTHS months up to the
    cut-off, without reversed postings (nor what they reverse) and without transfers between the client's own
    accounts (a debit and a credit with the same TRNS_RFR on the same client, netting to zero). The staging recipe
    calls it on postings grouped so that all the postings of a TRNS_RFR are together."""
    e = entries[entries["client_no"].isin(set(client_numbers))].copy()
    e["month"] = pd.to_datetime(e["value_date"]).dt.to_period("M").dt.to_timestamp()
    e = e[(e["month"] > cutoff - pd.DateOffset(months=HISTORY_MONTHS)) & (e["month"] <= cutoff)]
    marked = e["reversal"].notna() & ~e["reversal"].astype(str).str.strip().isin(["", "nan", "None"])
    e = e[~e["txn_ref"].isin(set(e.loc[marked, "txn_ref"].dropna()))]
    legs = e[e["txn_ref"].notna()].groupby(["txn_ref", "client_no"]).agg(sides=("direction", "nunique"),
                                                                         net=("amount", "sum"))
    own = legs[(legs["sides"] == 2) & (legs["net"].abs() < 0.005)].index
    e = e[~pd.MultiIndex.from_frame(e[["txn_ref", "client_no"]]).isin(own)]
    e = e.assign(amount=e["amount"].abs())
    return (e.groupby(["client_no", "month", "direction"], as_index=False)
            .agg(amount=("amount", "sum"), postings=("entry_id", "count")))


def set_cutoff():
    """The forecast's cut-off: the last complete month, by default the month before today. The project variable
    cutoff_override (YYYY-MM) fixes it instead: the sandbox's simulator sets it, and it reruns a past month. Sets the
    project variables cutoff_month, as_of_date and load_id (the data's as-of date), which the recipes read. The
    staging recipe stops if the postings do not reach the cut-off month (a late warehouse load).
    Run by the first step of each scenario: from warehouse import set_cutoff; set_cutoff()"""
    import dataiku
    project = dataiku.api_client().get_default_project()
    variables = project.get_variables()
    override = variables["standard"].get("cutoff_override")
    cutoff = pd.Period(override, "M") if override else pd.Timestamp.now().to_period("M") - 1
    as_of = cutoff.to_timestamp(how="end").normalize()
    variables["standard"].update(cutoff_month=str(cutoff), as_of_date=f"{as_of:%Y-%m-%d}",
                                 load_id=f"EDW_{as_of:%Y%m%d}")
    project.set_variables(variables)
    print(f"Cut-off month {cutoff} (data as of {as_of:%Y-%m-%d})" + (" (cutoff_override)" if override else ""))
    return str(cutoff)


for _table, _columns in COLUMNS.items():          # warehouse names unique within a table
    assert len(set(_columns.values())) == len(_columns), _table
'''

In [ ]:
LIBRARY_CODE = r'''
"""Shared by every zone (project library: python/cashflow.py): dates, the business rule on amounts, and the two
interfaces between the zones:
    1. the model rows (model_rows): staging -> training and inference. Training rows also have the ANSWER column.
       Add a column freely (a model that does not read it ignores it); rename or remove one only when training and
       inference both agree.
    3. the saved models (SAVED_MODELS): training adds versions, inference reads the active one.
Interface 2, the forecaster (lab -> training), is in models.py.
The forecast is one value per client and month: the net cash flow (money in minus money out), 1, 2 and 3 months
ahead, as P10, EV and P90. There are no categories."""
import numpy as np
import pandas as pd

MIN_HISTORY_MONTHS = 12     # a client needs this many months before the cut-off to get a forecast
HORIZONS = (1, 2, 3)        # months ahead
SAVED_MODELS = {"ev": "forecast_ev", "p10": "forecast_p10", "p90": "forecast_p90"}   # forecast column: saved model

# Interface 1, the model rows: one row per client, cut-off month and months ahead, with these columns. Training rows
# also have ANSWER (target: the net cash flow of the predicted month); scoring rows do not.
MODEL_ROW_COLUMNS = [
    "client_no", "cutoff_month", "months_ahead", "target_month", "history_months", "enough_history", "scale",
    "segment", "sector", "city", "sole_proprietor", "last_amount", "avg_3m", "avg_12m", "same_month_last_year",
    "client_in_avg_3m", "client_out_avg_3m", "target_month_of_year", "log_scale", "log_annual_sales", "employees"]
ANSWER = ["target"]


def read_dataset(name):
    """A dataset as a DataFrame, with dates as plain (time zone free) timestamps."""
    import dataiku
    df = dataiku.Dataset(name).get_dataframe()
    for c in df.columns:
        if isinstance(df[c].dtype, pd.DatetimeTZDtype):
            df[c] = df[c].dt.tz_convert(None)
    return df


def month_start(dates):
    """First day of the month, as a plain (time zone free) timestamp."""
    d = pd.to_datetime(dates)
    if getattr(d.dt, "tz", None) is not None:
        d = d.dt.tz_convert(None)
    return d.dt.to_period("M").dt.to_timestamp()


def months_between(start, end):
    """Whole months from start to end (both month timestamps)."""
    return (end.dt.year - start.dt.year) * 12 + (end.dt.month - start.dt.month)


def add_months(month, n):
    return month + pd.offsets.DateOffset(months=n)


def cutoff_month():
    """The forecast's cut-off: the last complete month (project variable
    cutoff_month, set by the first step of each scenario: warehouse.set_cutoff)."""
    import dataiku
    return pd.Timestamp(dataiku.get_custom_variables()["cutoff_month"] + "-01")


def finish(ev, p10, p90, index=None):
    """P10 <= EV <= P90 (SAR). The net cash flow can be below zero."""
    ev, p10, p90 = (np.asarray(v, dtype=float) for v in (ev, p10, p90))
    return pd.DataFrame({"p10": np.minimum(p10, ev), "ev": ev, "p90": np.maximum(p90, ev)}, index=index).round(2)


def model_rows(flows, clients, last_month):
    """Interface 1, the model rows: one row per client, cut-off month and months ahead (1, 2, 3), with the columns
    MODEL_ROW_COLUMNS and the answer (target: the net cash flow of the predicted month) where it is known. Training
    rows and scoring rows both come from here, so a model sees the same inputs in training and in the forecast. Inputs
    only use what was known at the end of the cut-off month.
    flows: client_no, month, direction (IN / OUT), amount (monthly_flows)
    clients: client_no, status, segment, sector, city, sole_proprietor, last_annual_sales, number_of_employees"""
    flows = flows.assign(month=month_start(flows["month"]))

    # ---- every client's months, zeros where nothing moved: money in, money out, net ----
    span = flows.groupby("client_no")["month"].agg(first_month="min", last_seen="max").reset_index()
    span = span.merge(clients[["client_no", "status"]], on="client_no")
    span["end_month"] = np.where(span["status"] == "ACTIVE", last_month, span["last_seen"])
    grid = pd.DataFrame([(r.client_no, m) for r in span.itertuples()
                         for m in pd.date_range(r.first_month, r.end_month, freq="MS")], columns=["client_no", "month"])
    sums = (flows.pivot_table(index=["client_no", "month"], columns="direction", values="amount", aggfunc="sum")
            .reindex(columns=["IN", "OUT"]).fillna(0.0).reset_index())
    series = grid.merge(sums, on=["client_no", "month"], how="left").rename(columns={"IN": "money_in", "OUT": "money_out"})
    series[["money_in", "money_out"]] = series[["money_in", "money_out"]].fillna(0.0)
    series["amount"] = series["money_in"] - series["money_out"]
    series = series.sort_values(["client_no", "month"]).reset_index(drop=True)

    # ---- inputs at each cut-off (the month itself is the cut-off) ----
    g = series.groupby("client_no")
    roll = lambda col, n: g[col].transform(lambda s: s.rolling(n, min_periods=1).mean())
    series["last_amount"] = series["amount"]
    series["avg_3m"], series["avg_12m"] = roll("amount", 3), roll("amount", 12)
    series["client_in_avg_3m"], series["client_out_avg_3m"] = roll("money_in", 3), roll("money_out", 3)
    series["scale"] = ((roll("money_in", 12) + roll("money_out", 12)) / 2).clip(lower=1000.0)
    for k in (9, 10, 11):
        series[f"lag_{k}"] = g["amount"].shift(k)
    base = series.merge(span[["client_no", "first_month"]], on="client_no")
    base["history_months"] = months_between(base["first_month"], base["month"]) + 1

    # ---- one row per months ahead, with its answer ----
    answers = series[["client_no", "month", "amount"]].rename(columns={"month": "target_month", "amount": "target"})
    profile = clients[["client_no", "segment", "sector", "city", "sole_proprietor", "last_annual_sales",
                       "number_of_employees"]]
    rows = []
    for h in HORIZONS:
        r = base.assign(months_ahead=h, target_month=base["month"].apply(lambda m: add_months(m, h)))
        r["same_month_last_year"] = r[f"lag_{12 - h}"]
        rows.append(r.merge(answers, on=["client_no", "target_month"], how="left"))
    rows = pd.concat(rows, ignore_index=True).rename(columns={"month": "cutoff_month"})
    rows = rows.merge(profile, on="client_no", how="left")
    rows["same_month_last_year"] = rows["same_month_last_year"].fillna(0.0)
    rows["target_month_of_year"] = rows["target_month"].dt.month
    rows["log_scale"] = np.log10(rows["scale"])
    rows["log_annual_sales"] = np.log10(rows["last_annual_sales"].astype(float).clip(lower=1.0))
    rows["employees"] = rows["number_of_employees"].astype(float)
    rows["enough_history"] = rows["history_months"] >= MIN_HISTORY_MONTHS
    return rows[MODEL_ROW_COLUMNS + ANSWER].reset_index(drop=True)
'''

In [ ]:
MODELS_CODE = r'''
"""Models (project library: python/models.py): interface 2, the forecaster, the reference forecaster (CatBoost) and the
test every model goes through. Training (train_and_test, champion_challenger) and the lab notebook use it; inference
never imports it.

A forecaster is a class with:
    name            its name. The project variable forecaster says which one the scenario Retrain, test and promote
                    trains (default: catboost).
    params          its settings (a dict), recorded in Experiment Tracking
    inputs          the model-row columns it reads (cashflow.MODEL_ROW_COLUMNS, no dates). Each saved model version
                    is given these columns.
    fit(rows)       learns from training rows. Column target is the amount to predict: the client's net cash flow
                    (money in minus money out) of the predicted month, in SAR.
    predict(rows)   a DataFrame with the index of rows and the columns p10, ev, p90 (SAR). forecast() then applies
                    P10 <= EV <= P90.
The data scientist's own forecasters go in forecasters.py: the setup notebook creates that file once and never
overwrites it. lab_try and lab_compare try forecasters against the production model without touching
production (the Lab zone and the notebook forecast_lab).
"""
import numpy as np
import pandas as pd

from cashflow import add_months, cutoff_month, finish, month_start

HOLDOUT_MONTHS = 6          # the last months kept aside to test the models
OUTPUT = ["p10", "ev", "p90"]

# the reference forecaster's inputs
CAT_FEATURES = ["segment", "sector", "city", "sole_proprietor"]
AMOUNT_FEATURES = ["last_amount", "avg_3m", "same_month_last_year", "client_in_avg_3m", "client_out_avg_3m"]
OTHER_FEATURES = ["months_ahead", "target_month_of_year", "history_months", "log_scale", "log_annual_sales",
                  "employees"]
FEATURES = CAT_FEATURES + AMOUNT_FEATURES + OTHER_FEATURES


def model_matrix(rows):
    """The reference forecaster's inputs: amounts divided by the client's scale (its usual month), categorical inputs
    as text."""
    X = rows[FEATURES].copy()
    for c in AMOUNT_FEATURES:
        X[c] = rows[c] / rows["scale"]
    for c in CAT_FEATURES:
        X[c] = X[c].fillna("NA").astype(str).str.upper()       # TRUE / FALSE however a yes / no column is read
    return X


class CatBoostForecaster:
    """The reference forecaster: one CatBoost model for the EV and one for P10 and P90 together, trained on the net
    cash flow divided by the client's scale (its usual month), so small and large clients look alike."""
    name = "catboost"
    params = {"iterations": 600, "depth": 6, "learning_rate": 0.08, "random_seed": 0, "verbose": 0}
    quantiles = (0.1, 0.9)
    inputs = FEATURES + ["scale"]

    def fit(self, rows):
        from catboost import CatBoostRegressor
        X, y = model_matrix(rows), rows["target"] / rows["scale"]
        self.ev = CatBoostRegressor(loss_function="RMSE", **self.params)
        self.ev.fit(X, y, cat_features=CAT_FEATURES)
        alphas = ",".join(str(q) for q in self.quantiles)
        self.p10_p90 = CatBoostRegressor(loss_function=f"MultiQuantile:alpha={alphas}", **self.params)
        self.p10_p90.fit(X, y, cat_features=CAT_FEATURES)
        return self

    def predict(self, rows):
        X = model_matrix(rows)
        scale = rows["scale"].values
        q = self.p10_p90.predict(X)
        return pd.DataFrame({"p10": q[:, 0] * scale, "ev": self.ev.predict(X) * scale, "p90": q[:, 1] * scale},
                            index=rows.index)


def known_forecasters():
    """The forecasters Retrain can train, by name: the reference one and those listed in FORECASTERS in
    forecasters.py. Also returns the problem reading forecasters.py, if any (the reference one still works)."""
    found, problem = {CatBoostForecaster.name: CatBoostForecaster}, None
    try:
        import forecasters
        for forecaster in forecasters.FORECASTERS:
            found[forecaster.name] = forecaster
    except Exception as e:
        problem = f"forecasters.py could not be read ({type(e).__name__}: {e})"
    return found, problem


def new_forecaster(name=None):
    """A new, untrained forecaster: the one named by the project variable forecaster (default catboost)."""
    import dataiku
    name = name or dataiku.get_custom_variables().get("forecaster") or CatBoostForecaster.name
    found, problem = known_forecasters()
    if name not in found:
        raise ValueError(f"No forecaster named {name!r} (project variable forecaster). Known: {', '.join(found)}. "
                         + (problem if problem else "Add yours to FORECASTERS in forecasters.py."))
    forecaster = found[name]()
    missing = [a for a in ("name", "inputs", "fit", "predict") if not hasattr(forecaster, a)]
    if missing:
        raise ValueError(f"Forecaster {name} has no {', '.join(missing)} (see models.py)")
    return forecaster


def forecast(forecaster, rows):
    """P10, EV and P90 of a trained forecaster for model rows (SAR), checked, with P10 <= EV <= P90. Everything that
    predicts goes through here: the test, champion / challenger, the lab and the saved models."""
    out = forecaster.predict(rows)
    name = getattr(forecaster, "name", type(forecaster).__name__)
    if not isinstance(out, pd.DataFrame) or not set(OUTPUT) <= set(out.columns):
        raise ValueError(f"Forecaster {name}: predict must return a DataFrame with the columns p10, ev, p90")
    if len(out) != len(rows) or not out.index.equals(rows.index):
        raise ValueError(f"Forecaster {name}: predict must return one row per input row, with the same index")
    empty = [c for c in OUTPUT if out[c].isna().any()]
    if empty:
        raise ValueError(f"Forecaster {name}: predict returned empty values in {', '.join(empty)}")
    return finish(out["ev"], out["p10"], out["p90"], rows.index)


def holdout(rows):
    """The test rows: predicted month in the last 6 months before the cut-off, with the simple guess (baseline: the
    average of the last 3 months)."""
    rows = rows.assign(target_month=month_start(rows["target_month"]))
    test = rows[rows["target_month"] >= add_months(cutoff_month(), -(HOLDOUT_MONTHS - 1))].copy()
    test["baseline"] = test["avg_3m"]
    return test


def scores(d):
    """Test scores of predictions (columns target, p10, ev, p90, baseline): EV error (WAPE) of the model and of the
    simple guess, and the share of actual values below P10, between P10 and P90, and above P90."""
    size = d["target"].abs().sum()
    wape = lambda column: round((d["target"] - d[column]).abs().sum() / size, 4) if size else np.nan
    return pd.Series({"rows": len(d), "actual_total": round(d["target"].sum(), 0), "wape_model": wape("ev"),
                      "wape_baseline": wape("baseline") if "baseline" in d else np.nan,
                      "coverage_p10_p90": round(((d["target"] >= d["p10"]) & (d["target"] <= d["p90"])).mean(), 4),
                      "below_p10": round((d["target"] < d["p10"]).mean(), 4),
                      "above_p90": round((d["target"] > d["p90"]).mean(), 4)})


def test_results(tested):
    """Test scores per months ahead (1, 2, 3) and overall (months_ahead ALL). tested: the test rows with p10, ev,
    p90."""
    scored = [c for c in ("target", "p10", "ev", "p90", "baseline") if c in tested]   # the columns scores reads
    results = pd.concat([
        tested.groupby("months_ahead")[scored].apply(scores).reset_index(),
        scores(tested).to_frame().T.assign(months_ahead="ALL")], ignore_index=True)
    results["months_ahead"] = results["months_ahead"].astype(str)
    return results


def run_metrics(results):
    """The scores recorded in Experiment Tracking: overall and the EV error per months ahead. results: from
    test_results."""
    overall = results[results["months_ahead"] == "ALL"].iloc[0]
    metrics = {k: overall[c] for k, c in (("wape", "wape_model"), ("wape_baseline", "wape_baseline"),
                                          ("coverage_p10_p90", "coverage_p10_p90"), ("below_p10", "below_p10"),
                                          ("above_p90", "above_p90"))}
    for row in results[results["months_ahead"] != "ALL"].itertuples():
        metrics[f"wape_{row.months_ahead}m"] = row.wape_model
    return metrics


def _lab_test():
    """The lab's test, the same as champion / challenger's: the training rows without the last 6 months (train), those
    months (test), the production model's ID and its backtest forecast on the test (None without a production model)."""
    import tracking
    from cashflow import read_dataset
    rows = read_dataset("training_rows")
    test = holdout(rows)
    production_id = tracking.production_model_id()
    production = test.join(forecast(tracking.load_backtest(production_id), test)) if production_id else None
    return rows.drop(test.index), test, production_id, production


def _by_months_ahead(tested):
    """Test scores per months ahead and overall (ALL), indexed by months ahead."""
    return test_results(tested).set_index("months_ahead")


def _record_try(forecaster, train, test, tested, production_id):
    """Record a lab try in Experiment Tracking (experiment cashflow_lab). Returns the run ID."""
    import tracking
    params = {"forecaster": forecaster.name, **getattr(forecaster, "params", {}),
              "cutoff_month": f"{cutoff_month():%Y-%m}", "training_rows": len(train), "test_rows": len(test),
              "compared_with": production_id or "none", "inputs": ",".join(forecaster.inputs)}
    run_name = f"{forecaster.name}_{pd.Timestamp.now():%Y%m%dT%H%M}"
    return tracking.log_lab_run(run_name, params, run_metrics(test_results(tested)), forecaster)


def lab_try(forecaster, record=True):
    """Try a forecaster against the production model without touching production (the lab: recipe try_forecaster;
    the notebook forecast_lab uses lab_compare). It is trained without the last 6 months and tested on them, the way
    champion / challenger tests a new model, and the try is recorded in Experiment Tracking (experiment cashflow_lab,
    never deployed).
    Returns: per months ahead (1, 2, 3, ALL) the EV error of the simple guess, of the production model and of the
    candidate, and their P10-P90 coverage; the decision champion / challenger would take and its reason; the lab run
    ID (None if not recorded)."""
    import tracking
    train, test, production_id, production = _lab_test()
    forecaster.fit(train)
    candidate = test.join(forecast(forecaster, test))
    mine = _by_months_ahead(candidate)
    table = pd.DataFrame({"simple_guess_error": mine["wape_baseline"], "production_error": np.nan,
                          "candidate_error": mine["wape_model"], "production_coverage": np.nan,
                          "candidate_coverage": mine["coverage_p10_p90"]})
    if production is not None:
        theirs = _by_months_ahead(production)
        table["production_error"], table["production_coverage"] = theirs["wape_model"], theirs["coverage_p10_p90"]
    decision, reason = tracking.decide(scores(candidate), scores(production) if production is not None else None)
    run_id = _record_try(forecaster, train, test, candidate, production_id) if record else None
    return table, decision, reason, run_id


def lab_compare(forecasters, record=True):
    """Try several forecasters at once against the production model, on the same test as lab_try and champion /
    challenger, without touching production. Each forecaster is trained without the last 6 months; each try is
    recorded in Experiment Tracking (experiment cashflow_lab) unless record is False. Returns a dict:
        board       one row per model (production first, simple guess last): EV error, P10-P90 coverage, share below
                    P10 and above P90, the decision champion / challenger would take and its reason, the lab run ID
        errors      EV error per months ahead: the simple guess, the production model and each forecaster
        coverage    P10-P90 coverage per months ahead: the production model and each forecaster
        tested      the test rows with each model's p10, ev, p90, by name, for your own analysis"""
    import tracking
    names = [f.name for f in forecasters]
    if len(set(names)) != len(names) or {"production", "simple_guess_baseline"} & set(names):
        raise ValueError(f"Give each forecaster its own name: {names}")
    train, test, production_id, production = _lab_test()
    tested = {} if production is None else {"production": production}
    for forecaster in forecasters:
        forecaster.fit(train)
        tested[forecaster.name] = test.join(forecast(forecaster, test))
    run_ids = {f.name: _record_try(f, train, test, tested[f.name], production_id) for f in forecasters} if record else {}
    reference = scores(production) if production is not None else None
    board = []
    for name, rows in tested.items():
        s = scores(rows)
        decision, reason = (("", f"the production model, {production_id}") if name == "production"
                            else tracking.decide(s, reference))
        board.append({"model": name, "ev_error": s["wape_model"], "coverage_p10_p90": s["coverage_p10_p90"],
                      "below_p10": s["below_p10"], "above_p90": s["above_p90"], "decision": decision,
                      "reason": reason, "lab_run_id": run_ids.get(name, "")})
    board.append({"model": "simple_guess_baseline", "ev_error": scores(next(iter(tested.values())))["wape_baseline"],
                  "decision": "", "reason": "average of the last 3 months: the bar to beat", "lab_run_id": ""})
    board = pd.DataFrame(board).set_index("model")
    per = {name: _by_months_ahead(rows) for name, rows in tested.items()}
    first = next(iter(per.values()))
    errors = pd.DataFrame({"simple_guess_baseline": first["wape_baseline"],
                           **{name: found["wape_model"] for name, found in per.items()}})
    coverage = pd.DataFrame({name: found["coverage_p10_p90"] for name, found in per.items()})
    return {"board": board, "errors": errors, "coverage": coverage, "tested": tested}
'''

In [ ]:
TRACKING_CODE = r'''
"""Model tracking and promotion (project library: python/tracking.py), for training and the lab.

Every training run is one model with its own model ID, recorded in Dataiku's Experiment Tracking (MLflow; artifacts in
the managed folder "experiments"): its forecaster and settings, the cut-off, test scores on the last 6 months, the
backtest forecaster (trained without those months, for champion / challenger tests) and three MLflow models (EV, P10,
P90). Those three are deployed as a new version (version ID = model ID) of the saved models forecast_ev, forecast_p10
and forecast_p90 (interface 3, cashflow.SAVED_MODELS). The production model is the active version; promoting a model
activates it in all three, and the Score recipes always use the active version.
Lab tries (notebook forecast_lab) go to their own experiment, cashflow_lab, in the folder lab_experiments. They are
never deployed.
"""
import inspect
import os
import pickle
import sys
import tempfile

import pandas as pd

from cashflow import SAVED_MODELS

EXPERIMENT, FOLDER = "cashflow_forecast", "experiments"
LAB_EXPERIMENT, LAB_FOLDER = "cashflow_lab", "lab_experiments"
MIN_COVERAGE, MAX_COVERAGE = 0.70, 0.90     # a model is only promoted if its P10-P90 coverage is in this range


def _project():
    import dataiku
    return dataiku.api_client().get_default_project()


def _mlflow(folder):
    """Experiment Tracking in the folder. By its ID: outside a recipe (the lab notebook), setup_mlflow does not resolve
    a folder name."""
    import dataiku
    return _project().setup_mlflow(managed_folder=dataiku.Folder(folder).get_id())


def _experiment_name(mlflow, experiment, folder):
    """The experiment, kept in the folder. If an experiment of that name points to another folder (the folder was
    deleted and created again, for example when the flow was deleted by hand), its files are gone: use
    <experiment>_<folder id> instead."""
    import dataiku
    folder_id = dataiku.Folder(folder).get_id()
    found = mlflow.get_experiment_by_name(experiment)
    if found is None or f"//{folder_id}/" in (found.artifact_location or ""):
        return experiment
    return f"{experiment}_{folder_id}"


def new_model_id(cutoff):
    """CFF_<cut-off month>_<training time>, for example CFF_202610_20261104T1215 (also the saved model version)."""
    return f"CFF_{cutoff:%Y%m}_{pd.Timestamp.now():%Y%m%dT%H%M}"


def saved_model(name):
    project = _project()
    return project.get_saved_model(next(m["id"] for m in project.list_saved_models() if m["name"] == name))


def forecast_model(forecaster, column):
    """An MLflow model around a trained forecaster: takes model rows, returns one forecast column (ev, p10 or p90)
    in SAR."""
    import mlflow.pyfunc
    from models import forecast

    class ForecastModel(mlflow.pyfunc.PythonModel):
        def __init__(self, forecaster, column):
            self.forecaster, self.column = forecaster, column

        def predict(self, context, model_input, params=None):
            return forecast(self.forecaster, model_input)[self.column].values

    return ForecastModel(forecaster, column)


def _carry_code(forecaster):
    """The models carry their own copy of the library code (pickled by value: cashflow, models, tracking and the
    forecaster's own module, forecasters.py for example), so they keep working when the library changes."""
    import cloudpickle
    import cashflow
    import models
    own = sys.modules.get(type(forecaster).__module__)
    for module in (cashflow, models, sys.modules[__name__], own):
        if module is not None and module.__name__ != "__main__":
            cloudpickle.register_pickle_by_value(module)


def _record(mlflow, params, metrics, backtest):
    """A run's settings, test scores and backtest forecaster."""
    import cloudpickle
    mlflow.log_params({k: str(v)[:250] for k, v in params.items()})
    mlflow.log_metrics({k: float(v) for k, v in metrics.items() if pd.notna(v)})
    with tempfile.TemporaryDirectory() as tmp:
        with open(os.path.join(tmp, "forecaster.pkl"), "wb") as f:
            cloudpickle.dump(backtest, f)
        mlflow.log_artifacts(tmp, artifact_path="backtest")


def log_model(model_id, params, metrics, backtest, final, reference_dataset):
    """Record one training run in Experiment Tracking, then deploy its final forecaster as version model_id of the
    three saved models (not activated: the promotion does that)."""
    _carry_code(final)
    with _mlflow(FOLDER) as mlflow:
        mlflow.set_experiment(_experiment_name(mlflow, EXPERIMENT, FOLDER))
        with mlflow.start_run(run_name=model_id) as run:
            mlflow.set_tag("model_id", model_id)
            _record(mlflow, params, metrics, backtest)
            where = "name" if int(mlflow.__version__.split(".")[0]) >= 3 else "artifact_path"   # MLflow 3 or 2
            for column in SAVED_MODELS:
                mlflow.pyfunc.log_model(python_model=forecast_model(final, column), **{where: column})
            run_id = run.info.run_id
    extension = _project().get_mlflow_extension()
    code_env = _project().get_recipe("train_and_test").get_settings().get_recipe_params()["envSelection"]["envName"]
    extension.set_run_inference_info(run_id=run_id, prediction_type="REGRESSION", code_env_name=code_env,
                                     target="target")
    # which model of the run: model_name in DSS 15 (MLflow 3 models), model_sub_folder in DSS 14 (the artifact path)
    which = "model_name" if "model_name" in inspect.signature(extension.deploy_run_model).parameters else "model_sub_folder"
    for column, saved_model_name in SAVED_MODELS.items():
        model = saved_model(saved_model_name)
        before = (model.get_active_version() or {}).get("id")
        # evaluated on the reference sample: gives the version its input schema (the Score recipes need it)
        extension.deploy_run_model(run_id, model.id, version_id=model_id,
                                   evaluation_dataset=reference_dataset, target_column_name="target",
                                   activate=False, skip_expensive_reports=True, **{which: column})
        if before and (model.get_active_version() or {}).get("id") != before:
            model.set_active_version(before)       # DSS 14 activates every deployed version: only a promotion may


def log_lab_run(run_name, params, metrics, backtest):
    """Record a lab try in Experiment Tracking (experiment cashflow_lab, folder lab_experiments): its settings, test
    scores and forecaster. Never deployed. Returns the run ID."""
    _carry_code(backtest)
    with _mlflow(LAB_FOLDER) as mlflow:
        mlflow.set_experiment(_experiment_name(mlflow, LAB_EXPERIMENT, LAB_FOLDER))
        with mlflow.start_run(run_name=run_name) as run:
            _record(mlflow, params, metrics, backtest)
            return run.info.run_id


def runs():
    """Every training run, newest first: model_id, run_id, trained_at, its settings (params.*) and metrics."""
    with _mlflow(FOLDER) as mlflow:
        # sorted here: DSS 14 cannot sort runs by start time in the search
        found = mlflow.search_runs(experiment_names=[_experiment_name(mlflow, EXPERIMENT, FOLDER)])
    if found.empty:
        return pd.DataFrame(columns=["model_id", "run_id", "trained_at"])
    found = found.sort_values("start_time", ascending=False).reset_index(drop=True)
    return found.rename(columns={"tags.model_id": "model_id", "start_time": "trained_at"})


def latest_model_id():
    """The most recently trained model."""
    found = runs()
    if found.empty:
        raise ValueError("No model trained yet: run the scenario Retrain, test and promote")
    return found.iloc[0]["model_id"]


def production_model_id():
    """The production model: the active version of the saved models, if it was ever promoted (a version Dataiku
    activated on its own when the first one was deployed does not count until the promotion)."""
    active = {name: (saved_model(sm).get_active_version() or {}).get("id") for name, sm in SAVED_MODELS.items()}
    if len(set(active.values())) != 1:
        raise ValueError(f"The saved models do not have the same active version: {active}. Activate the same model "
                         "ID in forecast_ev, forecast_p10 and forecast_p90.")
    model_id = active["ev"]
    return model_id if model_id in promoted_model_ids() else ""


def promoted_model_ids():
    """Every model ID promoted so far (read through the API: also works in a recipe that writes promotion_log)."""
    try:
        log = _project().get_dataset("promotion_log")
        columns = [c["name"] for c in log.get_schema()["columns"]]
        model, decision = columns.index("candidate_model_id"), columns.index("decision")
        return {row[model] for row in log.iter_rows() if row[decision] == "PROMOTE"}
    except Exception:            # no promotion yet
        return set()


def load_backtest(model_id):
    """The backtest forecaster of a training run (trained without the last 6 months, for champion / challenger)."""
    found = runs()
    match = found[found["model_id"] == model_id]
    if match.empty:
        raise ValueError(f"Model {model_id} not found in Experiment Tracking ({EXPERIMENT})")
    with _mlflow(FOLDER) as mlflow:
        path = mlflow.artifacts.download_artifacts(run_id=match.iloc[0]["run_id"], artifact_path="backtest")
    with open(os.path.join(path, "forecaster.pkl"), "rb") as f:
        return pickle.load(f)


def decide(candidate, production):
    """Champion / challenger: the decision (PROMOTE or KEEP) and its reason, from the test scores of the candidate and
    of the production model (models.scores). The candidate is promoted if its P10-P90 coverage is between 70% and 90%,
    it beats the simple guess, and its EV error is lower than the production model's (a tie keeps the production
    model). production None: no production model to compare with, the candidate is promoted (the forecast needs one)
    with its failed checks as a warning."""
    checks = []
    if not MIN_COVERAGE <= candidate["coverage_p10_p90"] <= MAX_COVERAGE:
        checks.append(f"coverage {candidate['coverage_p10_p90']:.1%} outside {MIN_COVERAGE:.0%}-{MAX_COVERAGE:.0%}")
    if candidate["wape_model"] >= candidate["wape_baseline"]:
        checks.append(f"does not beat the simple guess ({candidate['wape_model']:.1%} vs "
                      f"{candidate['wape_baseline']:.1%})")
    if production is None:
        return "PROMOTE", (f"no production model to compare with, EV error {candidate['wape_model']:.2%}, coverage "
                           f"{candidate['coverage_p10_p90']:.1%}" + (f" (warning: {'; '.join(checks)})" if checks else ""))
    if checks:
        return "KEEP", f"candidate {checks[0]}"
    if candidate["wape_model"] >= production["wape_model"]:
        return "KEEP", (f"candidate EV error {candidate['wape_model']:.2%} not lower than production "
                        f"{production['wape_model']:.2%}")
    return "PROMOTE", (f"candidate EV error {candidate['wape_model']:.2%} vs production {production['wape_model']:.2%}, "
                       f"coverage {candidate['coverage_p10_p90']:.1%}")


def apply_promotion():
    """Apply the latest decision in promotion_log: on PROMOTE, activate the candidate's version in the three saved
    models. Run by a scenario step after the champion / challenger test."""
    import dataiku
    log = dataiku.Dataset("promotion_log").get_dataframe()
    last = log.sort_values("decided_at").iloc[-1]
    if last["decision"] == "PROMOTE":
        for saved_model_name in SAVED_MODELS.values():
            saved_model(saved_model_name).set_active_version(last["candidate_model_id"])
        print(f"Promoted {last['candidate_model_id']}: {last['reason']}")
    else:
        print(f"Kept the production model: {last['reason']}")
    return production_model_id()
'''

In [ ]:
FORECAST_CODE = r'''
"""Inference (project library: python/forecast.py): the production model's ID and the final forecast table.
Inference reads nothing from training except the active version of the saved models (interface 3,
cashflow.SAVED_MODELS)."""
import numpy as np
import pandas as pd

from cashflow import HORIZONS, SAVED_MODELS, finish


def active_model_id():
    """The production model: the active version of the saved models forecast_ev, forecast_p10 and forecast_p90, the
    one the Score recipes use. All three must have the same."""
    import dataiku
    project = dataiku.api_client().get_default_project()
    ids = {m["name"]: m["id"] for m in project.list_saved_models()}
    missing = [name for name in SAVED_MODELS.values() if name not in ids]
    if missing:
        raise ValueError(f"Saved models not found: {', '.join(missing)}. Run the setup notebook.")
    active = {name: (project.get_saved_model(ids[name]).get_active_version() or {}).get("id")
              for name in SAVED_MODELS.values()}
    if not all(active.values()):
        raise ValueError("No production model yet: run the scenario Retrain, test and promote")
    if len(set(active.values())) != 1:
        raise ValueError(f"The saved models do not have the same active version: {active}. Activate the same model "
                         "ID in all three.")
    return next(iter(active.values()))


def forecast_table(pred, last_month):
    """The forecast: one row per client and month ahead, the net cash flow (money in minus money out) as P10, EV and
    P90 in SAR. pred: the model rows of the cut-off with the model's ev, p10, p90. A client with less than 12 months
    of history gets no forecast (status NOT_ENOUGH_HISTORY)."""
    frame = pred[["client_no", "months_ahead", "history_months", "enough_history", "p10", "ev", "p90"]].copy()
    frame[["p10", "ev", "p90"]] = finish(frame["ev"], frame["p10"], frame["p90"], frame.index)
    enough = frame["enough_history"].astype(bool)
    frame.loc[~enough, ["p10", "ev", "p90"]] = np.nan
    frame["cutoff_month"] = last_month
    frame["target_month"] = frame["months_ahead"].apply(lambda h: last_month + pd.offsets.DateOffset(months=h))
    frame["status"] = np.where(enough, "OK", "NOT_ENOUGH_HISTORY")
    return frame.sort_values(["client_no", "months_ahead"]).reset_index(drop=True)
'''

In [ ]:
FORECASTERS_CODE = r'''
"""The data scientist's forecasters (project library: python/forecasters.py). The setup notebook creates this file once
and never overwrites it: it is yours.

Try a forecaster first: in the notebook forecast_lab, or set the project variable lab_forecaster to its name and
build lab_results (Lab zone). To put one in production:
    1. add its class here and to FORECASTERS below,
    2. set the project variable forecaster to its name,
    3. run the scenario Retrain, test and promote: champion / challenger promotes it only if it beats the production
       model on the same last 6 months. To go back, set forecaster to catboost.

A forecaster is a class with (see models.py, and CatBoostForecaster there):
    name            the value of the project variable forecaster that selects it
    params          its settings (a dict), recorded in Experiment Tracking
    inputs          the model-row columns it reads (no dates)
    fit(rows)       learns from training rows (column target: the amount to predict, in SAR)
    predict(rows)   a DataFrame with the index of rows and the columns p10, ev, p90 (SAR)
"""
from models import CatBoostForecaster


class DeeperCatBoost(CatBoostForecaster):
    """Example: the reference CatBoost forecaster with deeper trees."""
    name = "catboost_deeper"
    params = {**CatBoostForecaster.params, "depth": 8}


FORECASTERS = [DeeperCatBoost]
'''

In [ ]:
RECIPES = {}

In [ ]:
RECIPES["stage_tables"] = r'''
# Staging: the 2 warehouse tables, read as they are (read-only, in chunks, only the columns used), turned into the
# project's tables, all in Dataiku:
#   clients: the business clients of DIM_CUSTOMER (companies and sole proprietors), standard names and values
#   monthly_flows: the postings of FCT_STATEMENT_ENTRY summed per client, month and direction (money in, money out):
#   business clients, the last 60 months up to the cut-off, no reversals, no transfers between the client's own
#   accounts. The postings are first spread over files on disk by TRNS_RFR, so that both sides of a
#   reversal or of an own-account transfer are in the same file, then summed one file at a time: memory stays small
#   however big the table is.
import os
import tempfile

import dataiku
import pandas as pd

from cashflow import cutoff_month
from warehouse import COLUMNS, HISTORY_MONTHS, OPTIONAL, TABLES, business_clients, monthly_flows, to_standard

CHUNK, FILES = 200_000, 32
cutoff = cutoff_month()


def read(table):
    """The table's chunks with the standard names, only the columns the project uses."""
    dataset = dataiku.Dataset(TABLES[table][0])
    there = {c["name"] for c in dataset.read_schema()}
    columns = [c for c in COLUMNS[table].values() if c in there or c not in OPTIONAL]
    for chunk in dataset.iter_dataframes(chunksize=CHUNK, columns=[c for c in columns if c in there]):
        yield to_standard(chunk, table)


clients = business_clients(pd.concat([business_clients(chunk) for chunk in read("clients")], ignore_index=True))
if clients.empty:
    raise ValueError("No business client in DIM_CUSTOMER: check the codes RETAIL and BUSINESS_ID in warehouse.py")
numbers = set(clients["client_no"])
dataiku.Dataset("clients").write_with_schema(clients)

first = cutoff - pd.DateOffset(months=HISTORY_MONTHS - 1)
with tempfile.TemporaryDirectory() as spread:
    read_rows = kept = 0
    for postings in read("transactions"):
        read_rows += len(postings)
        day = pd.to_datetime(postings["value_date"])
        postings = postings[postings["client_no"].isin(numbers) & (day >= first)
                            & (day < cutoff + pd.DateOffset(months=1))]
        kept += len(postings)
        part = pd.util.hash_pandas_object(postings["txn_ref"].fillna(postings["entry_id"].astype(str)),
                                          index=False) % FILES
        for n, rows in postings.groupby(part):
            path = os.path.join(spread, f"{n}.csv")
            rows.to_csv(path, mode="a", header=not os.path.exists(path), index=False)
    totals = []
    for name in sorted(os.listdir(spread)):
        rows = pd.read_csv(os.path.join(spread, name), dtype={"client_no": str, "txn_ref": str, "entry_id": str,
                                                              "reversal": str})
        totals.append(monthly_flows(rows, numbers, cutoff))
flows = pd.concat(totals, ignore_index=True) if totals else pd.DataFrame(columns=["client_no", "month"])
keys = ["client_no", "month", "direction"]
flows = flows.groupby(keys, as_index=False)[["amount", "postings"]].sum().round({"amount": 2})

# the postings must reach the cut-off month: none, or far fewer than the month before, means a late load
per_month = flows.groupby("month")["postings"].sum()
last, before = per_month.get(cutoff, 0), per_month.get(cutoff - pd.DateOffset(months=1), 0)
if last == 0 or last < 0.5 * before:
    raise ValueError(f"FCT_STATEMENT_ENTRY has {last:,} postings of business clients in {cutoff:%Y-%m} "
                     f"({before:,} the month before): the warehouse load for that month looks late or partial")
dataiku.Dataset("monthly_flows").write_with_schema(flows)
print(f"{len(clients):,} business clients; {read_rows:,} postings read, {kept:,} kept; {len(flows):,} monthly totals")
'''

In [ ]:
RECIPES["build_training_rows"] = r'''
# Training rows: one row per client, past cut-off month and months ahead (1, 2, 3), with its answer (the client's
# net cash flow in the predicted month). Training only: the nightly batch does not build it. The rows are made by model_rows in the project library (cashflow.py, interface 1),
# the same code that makes scoring_rows.
import dataiku

from cashflow import cutoff_month, model_rows, read_dataset

rows = model_rows(read_dataset("monthly_flows"), read_dataset("clients"), cutoff_month())
training = rows[rows["enough_history"] & rows["target"].notna()]
dataiku.Dataset("training_rows").write_with_schema(training.round(4).reset_index(drop=True))
'''

In [ ]:
RECIPES["train_and_test"] = r'''
# Train and test one model, with its own model ID, tracked in Experiment Tracking with its forecaster, settings and
# scores. The forecaster is the one the project variable forecaster names (default catboost, in models.py; the data
# scientist's own are in forecasters.py).
#   backtest: trained without the last 6 months, then tested on them (test_results)
#   final: trained on all rows, deployed as version <model ID> of the saved models forecast_ev, forecast_p10 and
#   forecast_p90 (not active: the promotion activates it). model_reference: a sample of the forecaster's inputs,
#   evaluated with each version so the version knows its input columns.
import dataiku

from cashflow import cutoff_month, read_dataset
from models import forecast, holdout, new_forecaster, run_metrics, test_results
from tracking import log_model, new_model_id

cutoff = cutoff_month()
rows = read_dataset("training_rows")
test = holdout(rows)
backtest, final = new_forecaster(), new_forecaster()
backtest.fit(rows.drop(test.index))
final.fit(rows)
model_id = new_model_id(cutoff)

results = test_results(test.join(forecast(backtest, test)))
results.insert(0, "model_id", model_id)
metrics = run_metrics(results)
inputs = list(final.inputs)
params = {"forecaster": final.name, **getattr(final, "params", {}), "cutoff_month": f"{cutoff:%Y-%m}",
          "load_id": dataiku.get_custom_variables().get("load_id", ""), "training_rows": len(rows),
          "test_rows": len(test), "clients": rows["client_no"].nunique(), "inputs": ",".join(inputs)}
reference = rows[inputs + ["target"]].sample(min(len(rows), 20000), random_state=0)
dataiku.Dataset("model_reference").write_with_schema(reference.reset_index(drop=True))
log_model(model_id, params, metrics, backtest, final, "model_reference")
print(f"Model {model_id} ({final.name}): EV error {metrics['wape']:.1%}, coverage {metrics['coverage_p10_p90']:.1%}")

dataiku.Dataset("test_results").write_with_schema(results)
'''

In [ ]:
RECIPES["champion_challenger"] = r'''
# Champion / challenger: the latest trained model (candidate) against the production model, both tested on the
# same last 6 months with their backtest forecasters (trained without those months). The rule is tracking.decide:
# promoted if its P10-P90 coverage is between 70% and 90%, it beats the simple guess (average of the last 3 months)
# and its EV error is lower than the production model's (a tie keeps the production model). If there is no production
# model to compare with (none yet, or its tracked files were deleted), the candidate is promoted (the forecast needs
# one), with its failed checks as a warning. The production model is the active version of the saved models. The
# decision is added to promotion_log; the next scenario step applies it (activates the candidate's version).
import dataiku
import pandas as pd

from cashflow import read_dataset
from models import forecast, holdout, scores
from tracking import decide, latest_model_id, load_backtest, production_model_id, runs

test = holdout(read_dataset("training_rows"))
candidate = latest_model_id()
production = production_model_id()
tracked = runs()
if production and production not in set(tracked["model_id"]):     # its tracked files were deleted: nothing to test
    print(f"Production model {production} is not in Experiment Tracking: the candidate is judged on its own")
    production = ""
tested = {m: scores(test.join(forecast(load_backtest(m), test))) for m in {candidate, production} if m}
c = tested[candidate]
p = tested.get(production) if production and production != candidate else None
if production == candidate:
    decision, reason = "KEEP", "the candidate is already the production model"
else:
    decision, reason = decide(c, p)
print(decision, "-", reason)

forecaster = dict(zip(tracked["model_id"], tracked["params.forecaster"])) if "params.forecaster" in tracked else {}
dataiku.Dataset("promotion_log").write_with_schema(pd.DataFrame([{
    "decided_at": pd.Timestamp.now().floor("s"), "cutoff_month": dataiku.get_custom_variables()["cutoff_month"],
    "candidate_model_id": candidate, "candidate_forecaster": forecaster.get(candidate),
    "production_model_id": production or None, "production_forecaster": forecaster.get(production),
    "candidate_wape": c["wape_model"], "production_wape": p["wape_model"] if p is not None else None,
    "baseline_wape": c["wape_baseline"], "candidate_coverage": c["coverage_p10_p90"],
    "production_coverage": p["coverage_p10_p90"] if p is not None else None,
    "decision": decision, "reason": reason}]))
'''

In [ ]:
RECIPES["build_scoring_rows"] = r'''
# Scoring rows: one row per client and months ahead (1, 2, 3) at the latest cut-off, to forecast with the
# production model. Inference only. The rows are made by model_rows in the project library (cashflow.py, interface 1),
# the same code that makes training_rows, so the model sees the same inputs it was trained on.
import dataiku

from cashflow import ANSWER, cutoff_month, model_rows, read_dataset

last_month = cutoff_month()
rows = model_rows(read_dataset("monthly_flows"), read_dataset("clients"), last_month)
scoring = rows[rows["cutoff_month"] == last_month].drop(columns=ANSWER)
dataiku.Dataset("scoring_rows").write_with_schema(scoring.round(4).reset_index(drop=True))
'''

In [ ]:
RECIPES["final_forecast"] = r'''
# Final forecast: the production model's prediction (the three Score recipes: EV, P10, P90, with the active version of
# the saved models): each client's net cash flow (money in minus money out) 1, 2 and 3 months ahead. Every row says
# which model (model_id: the active version) and which data (load_id) produced it. Inference reads nothing else from
# training.
import dataiku

from cashflow import cutoff_month, read_dataset
from forecast import active_model_id, forecast_table

keys = ["client_no", "months_ahead"]
pred = read_dataset("scored_ev").rename(columns={"prediction": "ev"})
for q in ("p10", "p90"):
    pred = pred.merge(read_dataset(f"scored_{q}")[keys + ["prediction"]].rename(columns={"prediction": q}), on=keys)
frame = forecast_table(pred, cutoff_month())
frame["model_id"] = active_model_id()
frame["load_id"] = dataiku.get_custom_variables().get("load_id", "")

cols = ["client_no", "cutoff_month", "months_ahead", "target_month", "p10", "ev", "p90", "history_months", "status",
        "model_id", "load_id"]
dataiku.Dataset("final_forecast").write_with_schema(frame[cols].round(2))
'''

In [ ]:
RECIPES["archive_forecast"] = r'''
# Archive and log each forecast run:
#   forecast_archive: one CSV file per cut-off month and model (forecast_<cut-off>_<model ID>.csv)
#   inference_log: one row per run (appended): when it ran, which model and load it used, what it produced
import dataiku
import pandas as pd

from cashflow import read_dataset

forecast = read_dataset("final_forecast")
first = forecast.iloc[0]
name = f"forecast_{first['cutoff_month']:%Y-%m}_{first['model_id']}.csv"
with dataiku.Folder("forecast_archive").get_writer(name) as w:
    w.write(forecast.to_csv(index=False).encode())

variables = dataiku.get_custom_variables()
clients = forecast.drop_duplicates("client_no")
log = pd.DataFrame([{
    "run_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "load_id": variables.get("load_id", ""),
    "as_of_date": variables.get("as_of_date", ""),
    "cutoff_month": f"{first['cutoff_month']:%Y-%m}",
    "model_id": first["model_id"],
    "clients": len(clients),
    "clients_not_enough_history": int((clients["status"] == "NOT_ENOUGH_HISTORY").sum()),
    "forecast_rows": len(forecast),
    "archive_file": name}])
dataiku.Dataset("inference_log").write_with_schema(log)
print("archived", name, "| logged run with model", first["model_id"])
'''

In [ ]:
RECIPES["try_forecaster"] = r'''
# Lab (data scientist): try a forecaster against the production model without touching production. The forecaster is
# the one the project variable lab_forecaster names (models.py or forecasters.py). It is trained without the last 6
# months and tested on them, the way champion / challenger tests a new model. The try is recorded in Experiment
# Tracking (experiment cashflow_lab, folder lab_experiments) and added to lab_results: per months ahead, its EV error and
# P10-P90 coverage next to the production model's and the simple guess, and the decision champion / challenger would
# take. Nothing here changes the saved models. Run it by building lab_results (only built when asked by name).
import dataiku
import pandas as pd

from cashflow import cutoff_month
from models import lab_try, new_forecaster

forecaster = new_forecaster(dataiku.get_custom_variables().get("lab_forecaster") or "catboost")
table, decision, reason, run_id = lab_try(forecaster)
print(f"{forecaster.name}: champion / challenger would decide {decision} ({reason})")

table = table.reset_index().assign(tried_at=pd.Timestamp.now().floor("s"), forecaster=forecaster.name,
                                   cutoff_month=f"{cutoff_month():%Y-%m}", decision=decision, reason=reason,
                                   lab_run_id=run_id)
cols = ["tried_at", "forecaster", "cutoff_month", "months_ahead", "simple_guess_error", "production_error",
        "candidate_error", "production_coverage", "candidate_coverage", "decision", "reason", "lab_run_id"]
dataiku.Dataset("lab_results").write_with_schema(table[cols])
'''

## 3. Build the flow

Uses the simulated warehouse tables, or links the warehouse's (read-only), and creates the recipes (Python and
Score) and their CSV datasets, the folders `experiments` (tracked
models), `forecast_archive` and `lab_experiments` (lab tries), the three saved models, and the project variables
`forecaster` (`catboost`) and `lab_forecaster` (`catboost_deeper`) if they are not set yet, in 5 zones. Training
(folder `experiments`), the promotion log, the saved models and the lab are only rebuilt when asked by name, so the
nightly batch never retrains by accident.

In [ ]:
import json
import sys
import time
import types

import pandas as pd

# Safe to run again in the same project: what already exists is kept (datasets and their data, folders, saved models and their versions, zones, the project variables forecaster and lab_forecaster,
# forecasters.py), and the code of the Python recipes and of the rest of the library is updated.
flow = project.get_flow()


def load_module(name, code):
    """A library file as a module of this notebook: the code it writes to the project library."""
    module = types.ModuleType(name)
    sys.modules[name] = module
    exec(code, module.__dict__)
    return module


cashflow = load_module("cashflow", LIBRARY_CODE)
warehouse = load_module("warehouse", WAREHOUSE_CODE)
tracking = load_module("tracking", TRACKING_CODE)
RAW = [warehouse_table for warehouse_table, _ in warehouse.TABLES.values()]
CSV = {"style": "excel", "charset": "utf8", "separator": ",", "quoteChar": '"', "escapeChar": "\\",
       "dateSerializationFormat": "ISO", "arrayMapFormat": "json", "parseHeaderRow": True, "compress": ""}

# Python recipes: (inputs, outputs, flow zone). Every dataset is a CSV file; experiments, forecast_archive and
# lab_experiments are folders. The visual recipes (Join, Group, Score) are created further down.
# Each zone has an owner, and the zones only meet through three interfaces:
#   1. the model rows (cashflow.model_rows): staging -> training_rows (Training) and scoring_rows (Inference)
#   2. the forecaster (models.py): lab -> training. The project variable forecaster names the one Retrain trains;
#      the Lab (try_forecaster, and the notebook forecast_lab) compares one with the production model, untouched.
#   3. the saved models' active version: training -> inference. Inference reads nothing else from training.
FLOW = {
    "stage_tables": (RAW, warehouse.STAGED, "Staging"),
    "build_training_rows": (["monthly_flows", "clients"], ["training_rows"], "Training"),
    "train_and_test": (["training_rows"], ["experiments", "test_results", "model_reference"], "Training"),
    "champion_challenger": (["training_rows", "experiments"], ["promotion_log"], "Training"),
    "build_scoring_rows": (["monthly_flows", "clients"], ["scoring_rows"], "Inference"),
    "final_forecast": (["scored_ev", "scored_p10", "scored_p90"], ["final_forecast"],
                       "Inference"),
    "archive_forecast": (["final_forecast"], ["forecast_archive", "inference_log"], "Inference"),
    "try_forecaster": (["training_rows", "experiments"], ["lab_experiments", "lab_results"], "Lab"),
}
FOLDERS = ["experiments", "forecast_archive", "lab_experiments"]
APPENDED = ["promotion_log", "inference_log", "lab_results"]      # logs: each run adds rows
BUILT_BY_NAME = ["experiments", "promotion_log", "lab_experiments", "lab_results"]   # only built when asked by name

try:                                                # needs the project's admin right
    metadata = project.get_metadata()
    metadata["label"] = PROJECT_NAME
    project.set_metadata(metadata)
except Exception as e:
    if not any(w in str(e).lower() for w in ("unauthorized", "forbidden", "permission", "not allowed")):
        raise
    print(f"Warning: could not name the project {PROJECT_NAME!r} (no admin right on the project): name it by hand")

# the project library: shared (cashflow.py, warehouse.py), training and lab (models.py, tracking.py), inference
# (forecast.py). forecasters.py is the data scientist's: written only when it is created, never overwritten.
library = project.get_library()
python_folder = library.get_folder("python") or library.add_folder("python")
for file_name, code in (("cashflow.py", LIBRARY_CODE), ("warehouse.py", WAREHOUSE_CODE), ("models.py", MODELS_CODE),
                        ("tracking.py", TRACKING_CODE), ("forecast.py", FORECAST_CODE)):
    (python_folder.get_file(file_name) or python_folder.add_file(file_name)).write(code)
if not python_folder.get_file("forecasters.py"):
    python_folder.add_file("forecasters.py").write(FORECASTERS_CODE)
    print("forecasters.py created (the data scientist's forecasters)")

# project variables, kept if they are already set: forecaster (the one Retrain trains), lab_forecaster (the one the
# recipe try_forecaster tries)
variables = project.get_variables()
defaults = {"forecaster": "catboost", "lab_forecaster": "catboost_deeper"}
missing = {k: v for k, v in defaults.items() if not variables["standard"].get(k)}
if missing:
    variables["standard"].update(missing)
    project.set_variables(variables)
print("Forecasters: Retrain trains {forecaster}, the lab tries {lab_forecaster}".format(**variables["standard"]))


# what the project already has (read once, kept up to date below)
datasets = {d["name"] for d in project.list_datasets()}
recipes = {r["name"] for r in project.list_recipes()}


def new_dataset(name):
    """Managed dataset of the project: a CSV file (comma-separated, with a header row) on CONNECTION. Kept if it
    exists."""
    if name in datasets:
        return name
    datasets.add(name)
    dataset = project.new_managed_dataset(name).with_store_into(CONNECTION).create()
    settings = dataset.get_settings()
    settings.get_raw()["formatType"] = "csv"
    settings.get_raw()["formatParams"].update(CSV)
    if name in BUILT_BY_NAME:
        settings.get_raw()["flowOptions"]["rebuildBehavior"] = "EXPLICIT"
    settings.save()
    return name


# the 2 warehouse tables: already here if the warehouse simulator wrote them (sandbox, CSV), otherwise read-only from
# the warehouse as they are (production): tables of WAREHOUSE_SQL_SCHEMA through the database connection
# WAREHOUSE_SQL_CONNECTION, or CSV files in the landing folder (LANDING_CONNECTION). Nothing is created there.
simulated = [name for name in RAW if name in datasets]
source = WAREHOUSE_SQL_CONNECTION or LANDING_CONNECTION
if WAREHOUSE_SQL_CONNECTION and len(simulated) < len(RAW):
    try:
        database = client.get_connection(WAREHOUSE_SQL_CONNECTION).get_info().get_type()   # Oracle, Teradata, ...
    except Exception as e:
        raise RuntimeError(f"Cannot use the connection {WAREHOUSE_SQL_CONNECTION!r} ({e}). Ask a Dataiku admin for a "
                           "read-only database connection to the warehouse, usable by your group, or check its "
                           "name in WAREHOUSE_SQL_CONNECTION.") from None
for name in RAW:
    if name in datasets:
        continue
    datasets.add(name)
    if WAREHOUSE_SQL_CONNECTION:
        project.create_sql_table_dataset(name, database, WAREHOUSE_SQL_CONNECTION, name, WAREHOUSE_SQL_SCHEMA)
    else:
        dataset = project.create_dataset(name, "Filesystem", params={"connection": LANDING_CONNECTION,
                                                                     "path": "/" + name},
                                          formatType="csv", formatParams=CSV)
        dataset.autodetect_settings().save()
print(f"{len(simulated)} warehouse tables already in the project, {len(RAW) - len(simulated)} read from {source}")

existing_folders = {f["name"]: f["id"] for f in project.list_managed_folders()}
folders = {}
for name in FOLDERS:
    if name in existing_folders:
        folders[name] = project.get_managed_folder(existing_folders[name])
        continue
    folders[name] = project.create_managed_folder(name, connection_name=FOLDER_CONNECTION)
    if name in BUILT_BY_NAME:
        folder_settings = folders[name].get_settings()
        folder_settings.get_raw()["flowOptions"]["rebuildBehavior"] = "EXPLICIT"
        folder_settings.save()
ref = lambda name: folders[name].id if name in folders else name
existing_models = {m["name"]: m["id"] for m in project.list_saved_models()}
saved_models = {name: project.get_saved_model(existing_models[name]) if name in existing_models
                else project.create_mlflow_pyfunc_model(name, "REGRESSION") for name in cashflow.SAVED_MODELS.values()}

for name, (inputs, outputs, _) in FLOW.items():
    for item in inputs + outputs:      # inputs made by recipes created later (Score) get their dataset now
        if item not in folders:
            new_dataset(item)
    if name in recipes:                # already there: only its code and code env are updated
        recipe = project.get_recipe(name)
    else:
        builder = project.new_recipe("python", name)
        for i in inputs:
            builder.with_input(ref(i))
        for o in outputs:
            builder.with_output(ref(o), append=o in APPENDED)
        recipe = builder.create()
        recipes.add(name)
    settings = recipe.get_settings()
    settings.set_payload(RECIPES[name])
    settings.set_code_env(code_env=CODE_ENV)
    settings.save()
    print("python recipe", name, "->", ", ".join(outputs))

# Score recipes: the active version of each saved model
for short, name in cashflow.SAVED_MODELS.items():
    if f"score_{short}" in recipes:
        continue
    builder = project.new_recipe("prediction_scoring", f"score_{short}")
    builder.with_input_model(saved_models[name].id)
    builder.with_input("scoring_rows")
    builder.with_existing_output(new_dataset(f"scored_{short}"))
    builder.create().rename(f"score_{short}")        # Dataiku names a Score recipe after its input
    print("score recipe", f"score_{short}", "->", f"scored_{short}")

ZONES = {"Raw data": RAW, "Staging": list(warehouse.STAGED), "Training": [], "Inference": ["scored_ev", "scored_p10", "scored_p90"], "Lab": []}
for _, outputs, zone_name in FLOW.values():
    ZONES[zone_name] += [o for o in outputs if o not in ZONES[zone_name]]
# the raw data goes in Dataiku's Default zone, renamed (so there is no empty zone). It exists once another zone does.
existing_zones = {z.name: z for z in flow.list_zones()}
zones = {z: existing_zones.get(z) or flow.create_zone(z) for z in ZONES if z != "Raw data"}
zones["Raw data"] = existing_zones.get("Raw data") or flow.get_default_zone()
if zones["Raw data"].name != "Raw data":
    zone_settings = zones["Raw data"].get_settings()
    zone_settings.name = "Raw data"
    zone_settings.save()


def safe_add(zone, item):
    """Put an item in a zone (already there: nothing to do)."""
    try:
        zone.add_item(item)
    except Exception as e:
        if "already" not in str(e).lower():
            raise


for zone_name, items in ZONES.items():
    for item in items:
        safe_add(zones[zone_name], folders[item] if item in folders else project.get_dataset(item))
for model in saved_models.values():       # next to the Score recipes that use them
    safe_add(zones["Inference"], model)
print("Flow ready:", ", ".join(zones))

## 4. First run

Checks the load, trains, tests and deploys a model, runs champion / challenger (with no production model yet, the
first model is promoted, with a warning in promotion_log if its checks fail), then runs the nightly batch: score,
forecast, archive, log.

In [ ]:
def build(items, mode):
    job = project.new_job(mode).with_auto_update_schema_before_each_recipe_run(True)
    for item in items:
        if item in folders:
            job.with_output(folders[item].id, object_type="MANAGED_FOLDER")
        else:
            job.with_output(item)
    job.start_and_wait()


started = time.time()
warehouse.set_cutoff()                                          # the first step of both scenarios
build(["training_rows"], "RECURSIVE_FORCED_BUILD")
build(["experiments"], "NON_RECURSIVE_FORCED_BUILD")         # train, test, track and deploy the first model
build(["promotion_log"], "NON_RECURSIVE_FORCED_BUILD")       # champion / challenger
production = tracking.apply_promotion()                        # activates it in the three saved models
build(["inference_log"], "RECURSIVE_FORCED_BUILD")           # score, forecast, archive, log the run
print(f"First run done in {(time.time() - started) / 60:.1f} minutes: production model {production}")

## 5. Scenarios

- **Nightly batch**: check the load, build the scoring rows, forecast with the production model, archive it and
  log the run. It does not touch the training data.
- **Retrain, test and promote**: check the load, prepare the training data, train, test and deploy a new model
  with the forecaster the project variable `forecaster` names, champion / challenger, activate it if better. It
  does not forecast: the next Nightly batch uses the new active version.

Their triggers (daily at 06:00 and monthly on the 5th, Riyadh time) are created switched off; switch them on
in production.

In [ ]:
# Production runs: two independent scenarios. Nightly batch (inference) forecasts with the active version of the saved
# models; Retrain, test and promote (training) adds a version and activates it if it is better, and never forecasts.
# Each starts by setting the cut-off month (a Python step); the staging recipe stops if the postings do not reach
# it. The tracked models (folder experiments) and the promotion log are only rebuilt when built by name, so the
# nightly batch never retrains by accident.
def step(n, label, what, mode=None):
    common = {"id": f"step_{n}", "name": label, "enabled": True, "runConditionType": "RUN_IF_STATUS_MATCH",
              "runConditionStatuses": ["SUCCESS", "WARNING"]}
    if isinstance(what, str):           # a Python step: library function to call
        module = "warehouse" if what == "set_cutoff" else "tracking"
        return {**common, "type": "custom_python", "params": {
            "script": f"from {module} import {what}\n\n{what}()\n", "envSelection": {"envMode": "INHERIT"},
            "proceedOnFailure": False, "handleWarningsAs": "WARNING"}}
    kind = lambda i: ("MANAGED_FOLDER", folders[i].id) if i in folders else ("DATASET", i)
    builds = [{"type": t, "itemId": item, "partitionsSpec": ""} for t, item in map(kind, what)]
    return {**common, "type": "build_flowitem", "params": {
        "builds": builds, "jobType": mode, "proceedOnFailure": False, "handleWarningsAs": "WARNING"}}


NIGHTLY = [
    ("Set the cut-off month", "set_cutoff"),
    ("Forecast with the production model, archive and log it", ["inference_log"], "RECURSIVE_FORCED_BUILD"),
]
SCENARIOS = {
    "Nightly batch": ("After each warehouse load: forecast the next 3 months with the production model (the active "
                      "version of the saved models), archive it and log the run", "daily", NIGHTLY),
    "Retrain, test and promote": (
        "Train and test a new model with the forecaster the project variable forecaster names (tracked in Experiment "
        "Tracking, deployed as a saved model version), and activate it if it beats the production model. It does not "
        "forecast: the Nightly batch uses the active version", "monthly", [
            ("Set the cut-off month", "set_cutoff"),
            ("Prepare the training data", ["training_rows"], "RECURSIVE_FORCED_BUILD"),
            ("Train, test and deploy a new model", ["experiments"], "NON_RECURSIVE_FORCED_BUILD"),
            ("Champion / challenger", ["promotion_log"], "NON_RECURSIVE_FORCED_BUILD"),
            ("Promote if better", "apply_promotion"),
        ]),
}
existing_scenarios = {s["name"]: s["id"] for s in project.list_scenarios()}
for name, (description, schedule, steps) in SCENARIOS.items():
    if name in existing_scenarios:     # run again: update its steps, keep its triggers and run history
        settings = project.get_scenario(existing_scenarios[name]).get_settings()
        settings.get_raw()["description"] = description
        settings.get_raw()["params"]["steps"] = [step(n, *s) for n, s in enumerate(steps)]
        settings.save()
        print(f"Scenario updated: {name}")
        continue
    scenario = project.create_scenario(name, "step_based", {
        "active": False, "description": description,
        "params": {"steps": [step(n, *s) for n, s in enumerate(steps)]}})
    settings = scenario.get_settings()
    if schedule == "daily":
        settings.add_daily_trigger(hour=6, timezone="Asia/Riyadh")
    else:
        settings.add_monthly_trigger(day=5, hour=7, timezone="Asia/Riyadh")
    settings.raw_triggers[-1]["active"] = False    # switch on in production (Automation node)
    try:
        settings.save()
        note = "trigger added, switched off"
    except Exception as e:                         # licences without time-based triggers (Free Edition)
        if "License" not in str(e):
            raise
        note = "no trigger (not in this licence): add it on the Automation node"
    print(f"Scenario ready: {name} ({note})")

## 6. Results

To use another model (for example to roll back), activate its version in the three saved models
(`forecast_ev`, `forecast_p10`, `forecast_p90`) and run the Nightly batch.

In [ ]:
print("Production model (active version of forecast_ev, forecast_p10, forecast_p90):", tracking.production_model_id())
print("Promotion log:")
display(dataiku.Dataset("promotion_log").get_dataframe())
print("Test on the last 6 months, net cash flow per client (by months ahead, and ALL):")
display(dataiku.Dataset("test_results").get_dataframe()[
    ["months_ahead", "rows", "wape_model", "wape_baseline", "coverage_p10_p90", "below_p10", "above_p90"]])
print("Inference log (one row per forecast run):")
display(dataiku.Dataset("inference_log").get_dataframe())